In [ ]:
# ==========================================
# 13 - OVERSAMPLING EXPERIMENT
# KHỞI TẠO MÔI TRƯỜNG
# ==========================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score
)

RANDOM_SEED = 42
N_SPLITS = 5

lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

output_dir = (
    ROOT_DIR
    / "experiments/v2_optimization/02_oversampling/results"
)

table_dir = output_dir / "tables"
config_dir = output_dir / "configs"

for directory in [table_dir, config_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Chỉ đọc Development
df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)
assert df_dev[id_col].is_unique
assert X.notna().all().all()
assert y.isin([0, 1]).all().all()
assert y.sum(axis=1).eq(4).all()

# Đọc 5 fold cố định
fold_data = []

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    assert set(train_ids).isdisjoint(set(val_ids))
    assert set(train_ids) | set(val_ids) == set(df_dev[id_col])

    train_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(train_ids).to_numpy()
    )

    val_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(val_ids).to_numpy()
    )

    fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

print("Development:", df_dev.shape)
print("X:", X.shape)
print("y:", y.shape)
print("Số fold:", len(fold_data))
print("Random Seed:", RANDOM_SEED)
print("PASS: Đã tải 5 fold cố định.")

In [ ]:
# ==========================================
# PHÂN TÍCH PHÂN BỐ TỔ HỢP TRONG TRAINING
# ==========================================

def encode_combinations(y_data):
    return [
        "".join(map(str, row))
        for row in y_data.to_numpy(dtype=int)
    ]


combo_records = []

for fold_info in fold_data:

    fold = fold_info["fold"]
    train_idx = fold_info["train_idx"]

    y_train = y.iloc[train_idx]

    combos = lib.pd.Series(
        encode_combinations(y_train)
    )

    counts = combos.value_counts()

    for combo, count in counts.items():
        combo_records.append({
            "Fold": fold,
            "Combination": combo,
            "Count": int(count),
            "Rate_%": round(
                count / len(y_train) * 100, 4
            )
        })

combo_distribution = lib.pd.DataFrame(combo_records)

print("=" * 60)
print("PHÂN BỐ TỔ HỢP MÔN TRONG TRAINING")
print("=" * 60)

display(
    combo_distribution.pivot(
        index="Combination",
        columns="Fold",
        values="Count"
    ).fillna(0).astype(int)
)

print("\nSố tổ hợp theo fold:")
display(
    combo_distribution.groupby("Fold")[
        "Combination"
    ].nunique().to_frame("Số tổ hợp")
)

In [ ]:
# ==========================================
# RANDOM OVERSAMPLING THEO TỔ HỢP NHÃN
# ==========================================

ROS_STRATEGIES = {
    "ros_mild": {
        "median_fraction": 0.25,
        "max_multiplier": 2
    },
    "ros_moderate": {
        "median_fraction": 0.50,
        "max_multiplier": 3
    }
}


def combination_random_oversampling(
    X_train,
    y_train,
    strategy,
    random_seed=42
):

    params = ROS_STRATEGIES[strategy]

    rng = lib.np.random.default_rng(random_seed)

    combo_codes = lib.np.array(
        encode_combinations(y_train)
    )

    counts = lib.pd.Series(
        combo_codes
    ).value_counts()

    # Trung vị tần suất các tổ hợp trong Training
    median_count = float(counts.median())

    target_count = int(
        lib.np.ceil(
            median_count * params["median_fraction"]
        )
    )

    # Giữ toàn bộ mẫu gốc
    sampled_positions = list(range(len(X_train)))

    sampling_records = []

    for combo, original_count in counts.items():

        positions = lib.np.flatnonzero(
            combo_codes == combo
        )

        # Không tăng quá hệ số nhân đã quy định
        desired_count = min(
            max(original_count, target_count),
            original_count * params["max_multiplier"]
        )

        n_extra = int(desired_count - original_count)

        if n_extra > 0:

            extra_positions = rng.choice(
                positions,
                size=n_extra,
                replace=True
            )

            sampled_positions.extend(
                extra_positions.tolist()
            )

        sampling_records.append({
            "Combination": combo,
            "Before": int(original_count),
            "After": int(desired_count),
            "Added": n_extra
        })

    sampled_positions = lib.np.array(
        sampled_positions,
        dtype=int
    )

    # X và y dùng chung vị trí để giữ đúng cặp dữ liệu
    X_resampled = (
        X_train.iloc[sampled_positions]
        .reset_index(drop=True)
    )

    y_resampled = (
        y_train.iloc[sampled_positions]
        .reset_index(drop=True)
    )

    sampling_report = lib.pd.DataFrame(
        sampling_records
    )

    return (
        X_resampled,
        y_resampled,
        sampling_report,
        sampled_positions
    )


# Kiểm tra thử trên Training của Fold 1
train_idx = fold_data[0]["train_idx"]

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

for strategy in ROS_STRATEGIES:

    X_res, y_res, report, positions = (
        combination_random_oversampling(
            X_train,
            y_train,
            strategy,
            RANDOM_SEED
        )
    )

    print(f"\nStrategy: {strategy}")
    print("Trước:", X_train.shape)
    print("Sau:", X_res.shape)

    display(report.sort_values("Before").head(10))

In [ ]:
# ==========================================
# KIỂM TRA OVERSAMPLING KHÔNG GÂY LEAKAGE
# ==========================================

sampling_log = []

for strategy in ROS_STRATEGIES:

    for fold_info in fold_data:

        fold = fold_info["fold"]
        train_idx = fold_info["train_idx"]
        val_idx = fold_info["val_idx"]

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_res, y_res, report, positions = (
            combination_random_oversampling(
                X_train,
                y_train,
                strategy,
                RANDOM_SEED + fold
            )
        )

        # Kiểm tra nguồn gốc mẫu
        original_ids = df_dev.iloc[
            train_idx
        ][id_col].to_numpy()

        resampled_ids = original_ids[positions]

        validation_ids = set(
            df_dev.iloc[val_idx][id_col]
        )

        assert set(resampled_ids).isdisjoint(
            validation_ids
        )

        assert set(resampled_ids).issubset(
            set(original_ids)
        )

        assert len(X_res) == len(y_res)

        sampling_log.append({
            "Strategy": strategy,
            "Fold": fold,
            "Original_Train": len(X_train),
            "Resampled_Train": len(X_res),
            "Added_Samples": len(X_res) - len(X_train),
            "Validation_Leakage": 0
        })

sampling_log_df = lib.pd.DataFrame(sampling_log)

display(sampling_log_df)

print("PASS: Tất cả mẫu nhân bản đều thuộc Training.")

In [ ]:
# ==========================================
# KIỂM TRA VALIDATION KHÔNG THAY ĐỔI
# ==========================================

validation_records = []

for fold_info in fold_data:

    fold = fold_info["fold"]
    val_idx = fold_info["val_idx"]

    val_ids = df_dev.iloc[
        val_idx
    ][id_col].tolist()

    saved_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig"
    )[id_col].tolist()

    assert set(val_ids) == set(saved_ids)
    assert len(val_ids) == len(saved_ids)

    validation_records.append({
        "Fold": fold,
        "Validation_Count": len(val_ids),
        "ID_Changed": 0,
        "Status": "PASS"
    })

validation_report = lib.pd.DataFrame(
    validation_records
)

display(validation_report)

print("PASS: Không thay đổi Validation của 5 fold.")

In [ ]:
# ==========================================
# KIỂM TRA RÀNG BUỘC MULTI-LABEL
# ==========================================

constraint_records = []

for strategy in ROS_STRATEGIES:

    for fold_info in fold_data:

        fold = fold_info["fold"]
        train_idx = fold_info["train_idx"]

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_res, y_res, report, positions = (
            combination_random_oversampling(
                X_train,
                y_train,
                strategy,
                RANDOM_SEED + fold
            )
        )

        binary_valid = y_res.isin([0, 1]).all().all()

        four_valid = y_res.sum(axis=1).eq(4).all()

        original_combos = set(
            encode_combinations(y_train)
        )

        new_combos = set(
            encode_combinations(y_res)
        )

        combos_valid = new_combos == original_combos

        assert binary_valid
        assert four_valid
        assert combos_valid

        constraint_records.append({
            "Strategy": strategy,
            "Fold": fold,
            "Samples": len(y_res),
            "Binary_Valid": binary_valid,
            "Exactly_4_Labels": four_valid,
            "Original_Combinations": len(original_combos),
            "Resampled_Combinations": len(new_combos),
            "Status": "PASS"
        })

constraint_report = lib.pd.DataFrame(
    constraint_records
)

display(constraint_report)

print("PASS: Oversampling bảo toàn ràng buộc 4/9 môn.")

In [ ]:
# ==========================================
# TRAIN 3 MODELS WITH OVERSAMPLING
# ==========================================

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

config_paths = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/random_forest/results/rf_config.json"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/xgboost/results/xgb_config.json"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/catboost/results/cat_config.json"
    )
}

v1_params = {}

for model_name, path in config_paths.items():

    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    v1_params[model_name] = config["hyperparameters"]


def create_model(model_name):

    params = v1_params[model_name].copy()

    # Không kết hợp Class Weight trong thí nghiệm này
    if model_name == "Random Forest":
        params["class_weight"] = None
        base_model = RandomForestClassifier(**params)

    elif model_name == "XGBoost":
        params["scale_pos_weight"] = 1.0
        base_model = XGBClassifier(**params)

    elif model_name == "CatBoost":
        params.pop("class_weights", None)
        params.pop("auto_class_weights", None)
        base_model = CatBoostClassifier(**params)

    else:
        raise ValueError("Mô hình không hợp lệ!")

    return MultiOutputClassifier(
        estimator=base_model,
        n_jobs=1
    )


def predict_positive_proba(model, X_val):

    probabilities = []

    for estimator in model.estimators_:

        classes = list(estimator.classes_)

        if 1 in classes:
            positive_index = classes.index(1)
            proba = estimator.predict_proba(
                X_val
            )[:, positive_index]
        else:
            proba = lib.np.zeros(len(X_val))

        probabilities.append(proba)

    return lib.np.column_stack(probabilities)


def predict_top4(probabilities):

    y_pred = lib.np.zeros_like(
        probabilities,
        dtype=int
    )

    top4_idx = lib.np.argsort(
        -probabilities,
        axis=1,
        kind="stable"
    )[:, :4]

    rows = lib.np.arange(
        len(probabilities)
    )[:, None]

    y_pred[rows, top4_idx] = 1

    return y_pred


experiment_results = []

# Chuẩn bị dữ liệu oversampling một lần cho mỗi fold/strategy
resampled_folds = {}

for strategy in ROS_STRATEGIES:

    for fold_info in fold_data:

        fold = fold_info["fold"]
        train_idx = fold_info["train_idx"]

        resampled_folds[(strategy, fold)] = (
            combination_random_oversampling(
                X.iloc[train_idx],
                y.iloc[train_idx],
                strategy,
                RANDOM_SEED + fold
            )[:2]
        )


for model_name in MODEL_NAMES:

    for strategy in ROS_STRATEGIES:

        print("=" * 60)
        print(f"{model_name} | {strategy}")
        print("=" * 60)

        for fold_info in fold_data:

            fold = fold_info["fold"]
            val_idx = fold_info["val_idx"]

            X_train_res, y_train_res = (
                resampled_folds[(strategy, fold)]
            )

            X_val = X.iloc[val_idx]
            y_val = y.iloc[val_idx]

            # Mô hình mới cho mỗi fold
            model = create_model(model_name)

            # Fit chỉ trên Training đã oversampling
            model.fit(
                X_train_res,
                y_train_res
            )

            # Predict trên Validation gốc
            y_proba = predict_positive_proba(
                model,
                X_val
            )

            y_pred = predict_top4(y_proba)

            assert y_proba.shape == (len(X_val), 9)
            assert lib.np.isfinite(y_proba).all()
            assert (y_pred.sum(axis=1) == 4).all()

            experiment_results.append({
                "Model": model_name,
                "Strategy": strategy,
                "Fold": fold,
                "val_idx": val_idx,
                "y_true": y_val.to_numpy(dtype=int),
                "y_pred": y_pred,
                "y_proba": y_proba
            })

            print(
                f"Fold {fold}: "
                f"Train={len(X_train_res):,} | "
                f"Validation={len(X_val):,} | PASS"
            )

print("\nPASS: Hoàn thành Oversampling Experiment.")
print("Số lượt đánh giá:", len(experiment_results))

In [ ]:
# ==========================================
# EVALUATION - OVERSAMPLING VS V1 & CLASS WEIGHT
# ==========================================

metric_records = []
label_records = []
prediction_frames = []

for result in experiment_results:

    info = {
        "Model": result["Model"],
        "Strategy": result["Strategy"],
        "Fold": result["Fold"]
    }

    y_true = result["y_true"]
    y_pred = result["y_pred"]

    metric_records.append({
        **info,
        "Exact_Match_Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Macro_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Micro_F1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "Hamming_Loss": hamming_loss(
            y_true, y_pred
        )
    })

    label_scores = f1_score(
        y_true, y_pred,
        average=None,
        zero_division=0
    )

    for label, score in zip(choice_cols, label_scores):
        label_records.append({
            **info,
            "Nhãn": label,
            "F1": score
        })

    pred_df = lib.pd.DataFrame({
        id_col: df_dev.iloc[
            result["val_idx"]
        ][id_col].to_numpy(),
        **info
    })

    for j in range(9):
        pred_df[f"true_{j+1}"] = y_true[:, j]
        pred_df[f"pred_{j+1}"] = y_pred[:, j]
        pred_df[f"proba_{j+1}"] = result["y_proba"][:, j]

    prediction_frames.append(pred_df)


ros_metrics = lib.pd.DataFrame(metric_records)
ros_label_f1 = lib.pd.DataFrame(label_records)

ros_predictions = lib.pd.concat(
    prediction_frames,
    ignore_index=True
)

# Đọc kết quả V1
v1_files = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/random_forest/results/rf_metrics_5fold.csv"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/xgboost/results/xgb_metrics_5fold.csv"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/catboost/results/cat_metrics_5fold.csv"
    )
}

v1_label_files = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/random_forest/results/rf_label_f1.csv"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/xgboost/results/xgb_label_f1.csv"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/catboost/results/cat_label_f1.csv"
    )
}

v1_metric_frames = []
v1_label_frames = []

for model_name in MODEL_NAMES:

    metrics = lib.pd.read_csv(v1_files[model_name])
    labels = lib.pd.read_csv(v1_label_files[model_name])

    metrics["Model"] = model_name
    metrics["Strategy"] = "V1_none"

    labels["Model"] = model_name
    labels["Strategy"] = "V1_none"

    v1_metric_frames.append(metrics)
    v1_label_frames.append(labels)

v1_metrics = lib.pd.concat(
    v1_metric_frames,
    ignore_index=True
)

v1_labels = lib.pd.concat(
    v1_label_frames,
    ignore_index=True
)

# Đọc Class Weight V2
weight_dir = ROOT_DIR / (
    "experiments/v2_optimization/01_class_weight/results/tables"
)

weight_metrics = lib.pd.read_csv(
    weight_dir / "v2_metrics_5fold.csv"
)

weight_labels = lib.pd.read_csv(
    weight_dir / "v2_label_f1.csv"
)

weight_metrics["Strategy"] = (
    "weight_" + weight_metrics["Strategy"]
)

weight_labels["Strategy"] = (
    "weight_" + weight_labels["Strategy"]
)

# Tổng hợp
all_metrics = lib.pd.concat(
    [v1_metrics, weight_metrics, ros_metrics],
    ignore_index=True
)

all_labels = lib.pd.concat(
    [v1_labels, weight_labels, ros_label_f1],
    ignore_index=True
)

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Hamming_Loss"
]

comparison = (
    all_metrics
    .groupby(["Model", "Strategy"])[metric_cols]
    .agg(["mean", "std"])
)

print("=" * 60)
print("SO SÁNH V1 - CLASS WEIGHT - OVERSAMPLING")
print("=" * 60)

display(comparison.round(4))

# So sánh F1 từng nhãn
label_comparison = (
    all_labels
    .groupby(["Model", "Strategy", "Nhãn"])["F1"]
    .mean()
    .reset_index()
)

for model_name in MODEL_NAMES:

    print(f"\nF1 THEO NHÃN - {model_name}")

    display(
        label_comparison[
            label_comparison["Model"] == model_name
        ].pivot(
            index="Nhãn",
            columns="Strategy",
            values="F1"
        ).round(4)
    )


# Lưu báo cáo
reports = {
    "oversampling_metrics_5fold.csv": ros_metrics,
    "oversampling_label_f1.csv": ros_label_f1,
    "oversampling_predictions.csv": ros_predictions,
    "sampling_log.csv": sampling_log_df,
    "combination_distribution.csv": combo_distribution,
    "validation_check.csv": validation_report,
    "constraint_check.csv": constraint_report,
    "v1_weight_ros_comparison.csv": comparison.reset_index(),
    "v1_weight_ros_label_f1.csv": label_comparison
}

for filename, data in reports.items():
    data.to_csv(
        table_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Lưu cấu hình thí nghiệm
experiment_config = {
    "experiment": "13 - Oversampling Experiment",
    "method": "Random Oversampling by Label Combination",
    "strategies": ROS_STRATEGIES,
    "models": MODEL_NAMES,
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "cv_source": "data/data-splits/cv_folds_v1",
    "training_source": "Development only",
    "validation_resampled": False,
    "class_weight_enabled": False,
    "prediction_strategy": "Top-4 probabilities",
    "hyperparameters": v1_params,
    "independent_test_used": False
}

with open(
    config_dir / "oversampling_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nPASS: Đã lưu kết quả Oversampling Experiment.")
print("Thư mục:", output_dir.resolve())
